# 03 — Three-arm comparison: what actually changed between 2022 and 2025?The collaborators observed that rebuilding their models in 2025 changed the phenotypepredictions, and worried the new models were worse. Two things changed **at the sametime**, so old-vs-new alone cannot say which was responsible:1. the genomes were **re-annotated** in KBase (bulk RAST), and2. the **modelling code** moved on by three years.A third arm separates them — rebuild from the **original PATRIC genomes** with the**current** code, holding the annotations fixed:| arm | annotations | build ||---|---|---|| **old** | PATRIC (2022) | KBase, 2022 || **new** | KBase re-annotation (2025) | KBase, 2025 || **rebuilt** | PATRIC (2022) | current (`MSBuilder`) |```old  vs rebuilt   same annotations, different build   ->  BUILD-CODE effectrebuilt vs new    same build, different annotations   ->  ANNOTATION effect```Every arm gets the **identical** downstream treatment (gapfill on `Carbon-Pyruvic-Acid`,simulate `ecoli_biolog` with CO2 blocked, threshold at 0.01) — otherwise the comparisonmeasures the treatment, not the models.**Scored against experiment, not against 2022.** Agreeing with the 2022 run only shows wereproduced a pipeline that we now know leaked CO2. Only the experimental Biolog data sayswhether a prediction is *right*.

## Step 1 — Load the sweepProduced by `scripts/three_arm_sweep.py`. Partial runs are fine — everything below reportswhat is present and says so.

In [ ]:
%run util.pysweep = pd.read_csv(NBOUTPUT_DIR / "three_arm_sweep.tsv", sep="\t")expected = 519 * 3print(f"{len(sweep)} of {expected} rows ({len(sweep)/expected:.0%})")if len(sweep) < expected:    print("*** PARTIAL -- numbers below cover the completed rows only ***")print()print(sweep["status"].value_counts().to_string())session.cache.save("three_arm_sweep", sweep)

## Step 2 — Model quality: how many models are thermodynamically broken?This is a finding in its own right, and it is not confined to this project — it is whatgapfilling does to almost any model built this way.- **free lunch** — biomass from **minerals alone**, no carbon of any kind. Impossible.  No media-level fix exists. Predictions meaningless. **Excluded from every number below**;  averaging them in would drag an arm's accuracy toward the base rate and look like a real  effect.- **autotrophic** — biomass from **CO2 + minerals**. Curable: block CO2 uptake (which the  media already specify) and it never fires. Kept; predictions sound.

In [ ]:
%run util.pysweep = session.cache.load("three_arm_sweep")rows = []for arm in ("old", "new", "rebuilt"):    a = sweep[sweep["arm"] == arm]    if a.empty:        continue    n = len(a)    fl = int((a["status"] == "free_lunch").sum())    auto = int(a[a["status"] == "ok"]["autotrophic"].fillna(0).astype(float).sum())    rows.append({"arm": arm, "n": n,                 "free_lunch": fl, "free_lunch_pct": f"{fl/n:.0%}",                 "autotrophic": auto, "autotrophic_pct": f"{auto/n:.0%}",                 "clean": n - fl - auto})quality = pd.DataFrame(rows)print(quality.to_string(index=False))NBOUTPUT_DIR.mkdir(parents=True, exist_ok=True)quality.to_csv(NBOUTPUT_DIR / "model_quality.tsv", sep="\t", index=False)session.cache.save("model_quality", quality)

## Step 3 — Accuracy against the experimental Biolog data

In [ ]:
%run util.pysweep = session.cache.load("three_arm_sweep")ok = sweep[sweep["status"] == "ok"].copy()for c in ("exp_accuracy", "clayton_accuracy"):    ok[c] = pd.to_numeric(ok[c], errors="coerce")acc = (ok.groupby("arm")         .agg(n=("exp_accuracy", "count"),              vs_experiment=("exp_accuracy", "mean"),              sd=("exp_accuracy", "std"),              vs_2022=("clayton_accuracy", "mean"))         .reindex(["old", "new", "rebuilt"]).dropna(how="all"))print(acc.round(4).to_string())acc.to_csv(NBOUTPUT_DIR / "arm_accuracy.tsv", sep="\t")session.cache.save("arm_accuracy", acc)

## Step 4 — The decompositionPaired **per genome**, so a genome that fails in one arm cannot skew the comparison, withstandard errors so we can say whether an effect is real or noise rather than readingmeaning into the third decimal place.**How to read it:** if the *total* old→new change is ~0 while the individual effects arenot, the two changes **cancelled** — accuracy held while the errors *moved*. That isexactly what Clayton reported and could not explain:> *"the success rate on the predictions remained about the same at approximately 70%, but a> lot of the patterns from the earlier set of models were lost and many of the explainable> inaccuracies are now seemingly random."*

In [ ]:
%run util.pysweep = session.cache.load("three_arm_sweep")ok = sweep[sweep["status"] == "ok"].copy()ok["exp_accuracy"] = pd.to_numeric(ok["exp_accuracy"], errors="coerce")wide = ok.pivot_table(index="genome_id", columns="arm", values="exp_accuracy")arms = [a for a in ("old", "new", "rebuilt") if a in wide.columns]paired = wide.dropna(subset=arms)print(f"paired on {len(paired)} genomes with a usable model in all {len(arms)} arms\n")def effect(a, b, label, meaning):    d = (paired[b] - paired[a]).dropna()    m = d.mean()    se = d.std() / len(d) ** 0.5 if len(d) > 1 else 0.0    verdict = "real (>2 SE)" if se and abs(m) > 2 * se else "within noise"    print(f"{label:20s} {m:+.4f} +/- {se:.4f} SE   n={len(d):4d}   {verdict}")    print(f"{'':20s} {meaning}\n")    return {"effect": label, "delta": m, "se": se, "n": len(d), "verdict": verdict}effects = []if {"old", "rebuilt"} <= set(arms):    effects.append(effect("old", "rebuilt", "BUILD-CODE effect",                          "same PATRIC annotations; 2022 build -> current build"))if {"rebuilt", "new"} <= set(arms):    effects.append(effect("rebuilt", "new", "ANNOTATION effect",                          "same modern build; PATRIC -> KBase re-annotation"))if {"old", "new"} <= set(arms):    effects.append(effect("old", "new", "TOTAL old -> new",                          "what the collaborators actually observed (both changes at once)"))effects_df = pd.DataFrame(effects)effects_df.to_csv(NBOUTPUT_DIR / "decomposition.tsv", sep="\t", index=False)session.cache.save("decomposition", effects_df)

## Step 5 — Where the errors movedAccuracy holding steady does not mean the predictions are the same. This counts theconditions whose **call flipped** between old and new — the "lost patterns" Claytondescribed, quantified.

In [ ]:
%run util.pysweep = session.cache.load("three_arm_sweep")ok = sweep[sweep["status"] == "ok"].copy()for c in ("exp_CP", "exp_CN", "exp_FP", "exp_FN"):    ok[c] = pd.to_numeric(ok[c], errors="coerce")# per-arm error profile: are the models over- or under-predicting growth?prof = (ok.groupby("arm")[["exp_CP", "exp_CN", "exp_FP", "exp_FN"]].mean()          .reindex(["old", "new", "rebuilt"]).dropna(how="all"))prof["over_predict(FP)"] = prof["exp_FP"]prof["under_predict(FN)"] = prof["exp_FN"]print("mean per-model error profile vs experiment:\n")print(prof[["exp_CP", "exp_CN", "over_predict(FP)", "under_predict(FN)"]].round(1).to_string())print("\nFN >> FP in every arm: these models UNDER-predict growth -- they miss real")print("substrates far more often than they invent false ones.")prof.to_csv(NBOUTPUT_DIR / "error_profile.tsv", sep="\t")